# 🚀 VLM LLaVA Training (Google Colab Edition)

This notebook runs the LLaVA training process for the MVTec Anomaly Detection dataset using Google Colab's GPU (T4 or A100).

### Prerequisites
1. **Upload Data**: Zip your local `mvtec_anomaly_detection` folder to `mvtec_anomaly_detection.zip` and upload it to `Project_Data` in your Google Drive.
2. **Upload JSON**: Upload `mvtec_train.json` to `Project_Data` in your Google Drive.
3. **Runtime**: Make sure to select **Runtime > Change runtime type > GPU**.

In [ ]:
# 1. Install Dependencies
!pip install --upgrade pip
# Downgrade NumPy to <2.0 to avoid binary incompatibility with older wheels
!pip install "numpy<2.0.0"
# Install dependencies (allowing pip to resolve compatible versions for Colab's Python 3.12)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.37.2 tokenizers==0.15.1 accelerate==0.27.2 peft==0.9.0 bitsandbytes>=0.43.0 gradio==4.16.0 shortuuid einops einops-exts timm openai-clip scikit-learn markdown2 protobuf sentencepiece requests pillow
!pip install flash-attn --no-build-isolation
!pip install deepspeed

In [ ]:
# 2. Clone LLaVA Repository
!git clone https://github.com/haotian-liu/LLaVA.git
%cd LLaVA
# Patch strict versions to allow installation on Colab (Python 3.12)
!sed -i 's/torch==2.1.2/torch>=2.1.2/g' pyproject.toml
!sed -i 's/torchvision==0.16.2/torchvision>=0.16.2/g' pyproject.toml
# Patch sentencepiece to avoid build errors (use newer binary wheel)
!sed -i 's/sentencepiece==/sentencepiece>=/g' pyproject.toml
!pip install -e .

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Prepare Dataset
import os
import shutil

# Define paths (Using Project_Data folder)
drive_zip_path = "/content/drive/MyDrive/Project_Data/mvtec_anomaly_detection.zip"
drive_json_path = "/content/drive/MyDrive/Project_Data/mvtec_train.json"

# Unzip Images to Runtime (Faster IO)
if os.path.exists(drive_zip_path):
    print("Unzipping dataset... (This may take a moment)")
    !unzip -qo "{drive_zip_path}" -d /content/LLaVA/mvtec_anomaly_detection
else:
    print(f"❌ Error: Zip file not found at {drive_zip_path}. Please check path.")

# Copy JSON
if os.path.exists(drive_json_path):
    shutil.copy(drive_json_path, "/content/LLaVA/mvtec_train.json")
    print("✅ JSON file copied.")
else:
    print(f"❌ Error: JSON file not found at {drive_json_path}. Please check path.")

In [ ]:
# 5. Run Training
import torch

# Configure Paths (Using local copies)
data_path = "/content/LLaVA/mvtec_train.json"
image_folder = "/content/LLaVA/mvtec_anomaly_detection"
output_dir = "/content/drive/MyDrive/Project_Data/llava-mvtec-checkpoints"

# Run LLaVA Training Script
!python llava/train/train_mem.py \
    --model_name_or_path liuhaotian/llava-v1.5-7b \
    --version v1 \
    --data_path {data_path} \
    --image_folder {image_folder} \
    --vision_tower openai/clip-vit-large-patch14-336 \
    --mm_projector_type mlp2x_gelu \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --image_aspect_ratio pad \
    --group_by_modality_length True \
    --bf16 True \
    --output_dir {output_dir} \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 500 \
    --save_total_limit 2 \
    --learning_rate 2e-4 \
    --weight_decay 0.0 \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --dataloader_num_workers 2 \
    --lazy_preprocess True \
    --report_to none \
    --lora_enable True \
    --lora_r 128 \
    --lora_alpha 256 \
    --lora_dropout 0.05 \
    --bits 4 \
    --double_quant True \
    --quant_type nf4

In [ ]:
# 6. Improved Evaluation: Load, Merge, and Test
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig
from peft import PeftModel
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from PIL import Image
import requests
import copy
import os
import warnings

warnings.filterwarnings("ignore")

# --- Configuration ---
base_model_path = "liuhaotian/llava-v1.5-7b"
output_dir = "/content/drive/MyDrive/Project_Data/llava-mvtec-checkpoints"

# Find latest checkpoint
lora_path = None
if os.path.exists(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split('-')[1]))
        lora_path = os.path.join(output_dir, checkpoints[-1])

print(f"🔹 Loading Base Model: {base_model_path}")
print(f"🔹 Loading LoRA Adapter: {lora_path}")

if lora_path:
    # Load Base Model
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, use_fast=False)
    # Use 4-bit loading for efficiency, similar to training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    # Initialise Vision Tower
    vision_tower = model.get_model().get_vision_tower()
    if not vision_tower.is_loaded:
        vision_tower.load_model()
    vision_tower.to(device="cuda", dtype=torch.float16)
    
    # Load LoRA Adapters
    print("Merging Low-Rank Adapters...")
    model = PeftModel.from_pretrained(model, lora_path)
    
    print("✅ Model successfully loaded!")

    # Inference Helper Function
    def eval_mvtec(image_file, prompt="Is there any anomaly in this image? Describe it."):
        if not os.path.exists(image_file):
            print(f"❌ File not found: {image_file}")
            return

        # Process Image
        image = Image.open(image_file).convert('RGB')
        image_tensor = process_images([image], model.get_model().vision_tower.image_processor, model.config)
        if type(image_tensor) is list:
            image_tensor = [image.to(model.device, dtype=torch.float16) for image in image_tensor]
        else:
            image_tensor = image_tensor.to(model.device, dtype=torch.float16)

        # Prepare Prompt
        conv = conv_templates["v1"].copy()
        inp = DEFAULT_IMAGE_TOKEN + "\n" + prompt
        conv.append_message(conv.roles[0], inp)
        conv.append_message(conv.roles[1], None)
        prompt_str = conv.get_prompt()

        input_ids = tokenizer_image_token(prompt_str, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()

        # Generate Response
        with torch.inference_mode():
            output_ids = model.generate(
                input_ids,
                images=image_tensor,
                do_sample=True,
                temperature=0.2,
                max_new_tokens=512,
                use_cache=True
            )

        outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
        
        display(image.resize((224, 224))) # Show thumbnail
        print(f"❓ Question: {prompt}")
        print(f"💡 Answer: {outputs}\n" + "-"*50)

    # --- Run Tests ---
    import glob
    image_folder = "/content/LLaVA/mvtec_anomaly_detection"

    # 1. Test 'Good' Image (No Anomaly)
    good_images = glob.glob(f"{image_folder}/bottle/test/good/*.png")
    if good_images:
        print("Testing GOOD sample:")
        eval_mvtec(good_images[0], "Is there any anomaly in this image? Answer Yes or No.")
        
    # 2. Test 'Anomaly' Image (Broken)
    bad_images = glob.glob(f"{image_folder}/bottle/test/broken_large/*.png")
    if bad_images:
        print("Testing ANOMALY sample:")
        eval_mvtec(bad_images[0], "Is there any anomaly in this image? Describe it.")

else:
    print("❌ No checkpoint found to load.")